# Approximation algorithms for the knapsack problem (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/knapsack-approximation_python.ipynb)

Executable notebook of the chapter “Approximation algorithms for the knapsack problem” (part “Approximation algorithms” of the course *Combinatorial Optimization*): it contains the approximation algorithms and schemes of the chapter for the knapsack problem: the greedy algorithm, Ext-Greedy, $H^2$, $H^{\ell}$, dynamic programming by profits and Basic-FPTAS, written in Python and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/approximation-algorithms/knapsack-approximation/). The same notebook is also available in [C++](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/knapsack-approximation_cpp.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`: the cells must be run in order, since each one uses the functions defined above it. Nothing to install: only the Python standard library is used.

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## The greedy algorithm

The items are assumed to be sorted by non-increasing efficiency.

In [1]:
# Approximation algorithms and schemes for the knapsack problem: items are indexed 0, 1, ..., n-1
# and are assumed to be sorted by non-increasing efficiency.
# Every function returns the pair (z, x): the profit of the solution found and the list x
# (x[i] = 1 if item i is selected, 0 otherwise); the C++ version fills an output array x.

def greedy_kp(n, c, w, p):
    """Greedy algorithm GR (items sorted by non-increasing efficiency).
    Returns (z, x): the profit of the solution and the solution x."""
    x = [0] * n                 # every item is initially not selected
    rc = c                      # residual capacity

    for i in range(n):
        if w[i] <= rc:
            x[i] = 1            # select item i
            rc = rc - w[i]      # update residual capacity

    # profit of the solution x, returned for convenience
    z = 0
    for i in range(n):
        if x[i] == 1:
            z = z + p[i]
    return z, x

## The Ext-Greedy algorithm

The better of the greedy solution and of the most profitable item alone.

In [2]:
def extended_greedy_kp(n, c, w, p):
    """Ext-Greedy algorithm GR^e (items sorted by non-increasing efficiency).
    Returns (z, x): the profit of the solution and the solution x."""
    z, x = greedy_kp(n, c, w, p)    # greedy solution and its value

    im = 0                          # best single item
    pm = p[0]
    for i in range(1, n):
        if p[i] > pm:
            im = i                  # update best single item
            pm = p[i]               # update its profit

    xhat = [0] * n                  # best single-item solution
    xhat[im] = 1                    # select only item im

    if z < pm:
        for i in range(n):
            x[i] = xhat[i]          # copy component by component
        z = pm

    return z, x

## The $H^2$ algorithm

It calls Ext-Greedy on the residual instance of every feasible pair, exactly as the pseudo-code does, and it keeps the best solution found.

In [3]:
def h2_kp(n, c, w, p):
    """Algorithm H^2 (items sorted by non-increasing efficiency).
    Returns (lb, x): H2(I), the value of the best solution found, and that solution x,
    which the pseudo-code does not record."""
    im = 0
    for j in range(1, n):
        if p[j] > p[im]:
            im = j
    pmax = p[im]
    lb = pmax                                   # best value found
    x = [0] * n                                 # record the solution
    x[im] = 1

    for i in range(n):
        for k in range(i + 1, n):
            if w[i] + w[k] > c:
                continue                        # the pair (i,k) is not in P

            cbar = c - w[i] - w[k]              # residual capacity
            pmin = p[i] if p[i] < p[k] else p[k]

            # residual set I(i,k): original indices, profits and weights of the residual instance I_ik
            idx = []
            pr = []
            wr = []
            for j in range(n):
                if j != i and j != k and p[j] <= pmin and w[j] <= cbar:
                    idx.append(j)
                    pr.append(p[j])
                    wr.append(w[j])
            m = len(idx)

            # value of GR^e on the residual instance
            if m > 0:
                zres, xr = extended_greedy_kp(m, cbar, wr, pr)
            else:
                zres, xr = 0, []
            lbik = p[i] + p[k] + zres

            if lbik > lb:
                lb = lbik
                x = [0] * n                     # record the solution
                x[i] = 1
                x[k] = 1
                for j in range(m):
                    if xr[j] == 1:
                        x[idx[j]] = 1

    return lb, x

## The $H^{\ell}$ algorithm

`NextSubset` enumerates the subsets of exactly $r$ items in lexicographic order; the preprocessing examines the subsets of at most $\ell-1$ items, and each subset of exactly $\ell$ items is completed by Ext-Greedy on its residual instance, as in the pseudo-code.

In [4]:
def next_subset(L, r, n):
    """Next subset of r indices of {0,...,n-1} in lexicographic order:
    L is modified in place; returns False at the end."""
    t = r - 1
    while t >= 0 and L[t] == n - r + t:
        t -= 1
    if t < 0:
        return False
    L[t] += 1
    for u in range(t + 1, r):
        L[u] = L[u - 1] + 1
    return True

In [5]:
def hl_kp(n, c, w, p, l):
    """Algorithm H^l, with 2 <= l <= n guessed items (items sorted by non-increasing efficiency).
    Returns (lb, x): H^l(I), the value of the best solution found, and that solution x,
    which the pseudo-code does not record."""
    L = [0] * l

    # lb := best value of the feasible sets of at most l-1 items
    lb = 0
    x = [0] * n
    for r in range(1, l):
        for t in range(r):
            L[t] = t
        while True:
            wS = 0
            pS = 0
            for t in range(r):
                wS += w[L[t]]
                pS += p[L[t]]
            if wS <= c and pS > lb:
                lb = pS
                x = [0] * n                     # record the solution
                for t in range(r):
                    x[L[t]] = 1
            if not next_subset(L, r, n):
                break

    # foreach L in T(l): the feasible sets of exactly l items
    for t in range(l):
        L[t] = t
    while True:
        wL = 0
        pL = 0
        pmin = p[L[0]]
        inL = [0] * n
        for t in range(l):
            wL += w[L[t]]
            pL += p[L[t]]
            inL[L[t]] = 1
            if p[L[t]] < pmin:
                pmin = p[L[t]]

        if wL <= c:                             # otherwise L is not in T(l)
            cbar = c - wL                       # residual capacity

            # residual set I(L)
            idx = []
            pr = []
            wr = []
            for j in range(n):
                if not inL[j] and p[j] <= pmin and w[j] <= cbar:
                    idx.append(j)
                    pr.append(p[j])
                    wr.append(w[j])
            m = len(idx)

            # complete the guess
            if m > 0:
                zres, xr = extended_greedy_kp(m, cbar, wr, pr)
            else:
                zres, xr = 0, []
            lbL = pL + zres

            if lbL > lb:
                lb = lbL
                x = inL[:]                      # record the solution
                for j in range(m):
                    if xr[j] == 1:
                        x[idx[j]] = 1

        if not next_subset(L, l, n):
            break

    return lb, x

## Dynamic programming by profits

The second DP algorithm of the dynamic programming chapters, used by Basic-FPTAS. The table `u[i][q]` stores $u(i,q)$, and an optimal solution is reconstructed backwards. The parameter `qmax` truncates the table: only the profits $q\le\min\{P, qmax\}$ are computed, where $P$ is the total profit.

In [6]:
def dp_profits(n, c, w, p, qmax):
    """Dynamic programming by profits (second DP algorithm of the dynamic programming notes).
    u(i,q) = minimum total weight of a subset of {0,...,i-1} with total profit exactly q;
    the table is truncated at the profit qmax: only q = 0,...,min{P, qmax} are computed,
    where P is the total profit; qmax must be at least the optimal value of the instance.
    Returns (qstar, x): the optimal value, i.e., the largest q with u(n,q) <= c,
    and an optimal solution x."""
    INF = 2147483647
    P = 0                                       # total profit
    for i in range(n):
        P += p[i]
    if P > qmax:
        P = qmax                                # truncation of the table

    # table u[i][q], i = 0,...,n and q = 0,...,P (P = min{total profit, qmax})
    u = [[0] * (P + 1) for i in range(n + 1)]

    u[0][0] = 0
    for q in range(1, P + 1):
        u[0][q] = INF                           # no item: only q = 0 is attained

    for i in range(1, n + 1):
        for q in range(P + 1):
            u[i][q] = u[i - 1][q]               # item i-1 is not selected
            if (p[i - 1] <= q and u[i - 1][q - p[i - 1]] != INF
                    and w[i - 1] + u[i - 1][q - p[i - 1]] < u[i][q]):
                u[i][q] = w[i - 1] + u[i - 1][q - p[i - 1]]   # item i-1 is selected

    # optimal value: the largest profit attained within the capacity
    qstar = 0
    for q in range(P + 1):
        if u[n][q] <= c:
            qstar = q

    # reconstruct an optimal solution backwards
    x = [0] * n
    q = qstar
    for i in range(n, 0, -1):
        if u[i][q] == u[i - 1][q]:
            x[i - 1] = 0
        else:
            x[i - 1] = 1
            q -= p[i - 1]

    return qstar, x

## The Basic-FPTAS algorithm

The accuracy $\varepsilon$ is passed as a fraction $\varepsilon_{num}/\varepsilon_{den}$, so that the scaled profits $\lfloor p_j/K\rfloor$ are computed in exact integer arithmetic, and the table of `DPProfits` is truncated at $\lfloor 2n/\varepsilon\rfloor$.

In [7]:
def basic_fptas(n, c, w, p, eps_num, eps_den):
    """Basic-FPTAS with accuracy eps = eps_num / eps_den, 0 < eps < 1
    (items sorted by non-increasing efficiency).
    Returns (z, x): the profit, in the original instance, of the solution x."""
    lb, x = extended_greedy_kp(n, c, w, p)      # value of the Ext-Greedy solution

    # scaling factor K = eps * lb / n and scaled profits p'[j] = floor(p[j] / K);
    # with eps = eps_num / eps_den, p[j] / K = p[j] * n * eps_den / (eps_num * lb),
    # which is computed in exact integer arithmetic
    ps = [0] * n                                # scaled profits p'
    for j in range(n):
        ps[j] = (p[j] * n * eps_den) // (eps_num * lb)

    # the optimal value of the scaled instance is at most floor(2n / eps): the table is truncated there
    qmax = (2 * n * eps_den) // eps_num

    zs, x = dp_profits(n, c, w, ps, qmax)       # scaled instance solved to optimality

    # profit of the solution x in the original instance
    z = 0
    for j in range(n):
        if x[j] == 1:
            z = z + p[j]

    return z, x

## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [8]:
# runs every example of the lecture notes and every instance of the exercises,
# and compares the value found with the expected one

errors = 0
checks = 0


def check(instance, algorithm, found, expected):
    global errors, checks
    checks += 1
    if found != expected:
        errors += 1
    print("%-22s %-14s %4d %s" % (instance, algorithm, found,
                                  "OK" if found == expected else "<-- EXPECTED A DIFFERENT VALUE"))

p = [6, 5, 8, 9, 6, 7, 3]
w = [2, 3, 6, 7, 5, 9, 4]
check("Example 1 (c=9)", "GR", greedy_kp(7, 9, w, p)[0], 14)
check("Example 1 (c=9)", "GRe", extended_greedy_kp(7, 9, w, p)[0], 14)
check("Example 1 (c=9)", "H2", h2_kp(7, 9, w, p)[0], 15)
check("Example 1 (c=9)", "Hl^2", hl_kp(7, 9, w, p, 2)[0], 15)
check("Example 1 (c=9)", "Hl^3", hl_kp(7, 9, w, p, 3)[0], 15)

p = [6, 8, 19, 4]
w = [2, 3, 9, 5]
check("Example 2", "GR", greedy_kp(4, 10, w, p)[0], 18)
check("Example 2", "GRe", extended_greedy_kp(4, 10, w, p)[0], 19)
check("Example 2", "H2", h2_kp(4, 10, w, p)[0], 19)
check("Example 2", "Hl^2", hl_kp(4, 10, w, p, 2)[0], 19)
check("Example 2", "Hl^3", hl_kp(4, 10, w, p, 3)[0], 19)

p = [4, 6, 5, 12, 3]
w = [1, 2, 2, 6, 2]
check("Example 3", "GR", greedy_kp(5, 10, w, p)[0], 18)
check("Example 3", "GRe", extended_greedy_kp(5, 10, w, p)[0], 18)
check("Example 3", "H2", h2_kp(5, 10, w, p)[0], 23)
check("Example 3", "Hl^2", hl_kp(5, 10, w, p, 2)[0], 23)
check("Example 3", "Hl^3", hl_kp(5, 10, w, p, 3)[0], 23)

p = [4, 6, 4, 3, 2, 20]
w = [1, 2, 2, 2, 2, 21]
check("Example 4", "GR", greedy_kp(6, 21, w, p)[0], 19)
check("Example 4", "GRe", extended_greedy_kp(6, 21, w, p)[0], 20)
check("Example 4", "H2", h2_kp(6, 21, w, p)[0], 20)
check("Example 4", "Hl^2", hl_kp(6, 21, w, p, 2)[0], 20)
check("Example 4", "Hl^3", hl_kp(6, 21, w, p, 3)[0], 20)

p = [6, 5, 8, 9, 6, 7, 3]
w = [2, 3, 6, 7, 5, 9, 4]
check("Example H2 (c=12)", "GR", greedy_kp(7, 12, w, p)[0], 19)
check("Example H2 (c=12)", "GRe", extended_greedy_kp(7, 12, w, p)[0], 19)
check("Example H2 (c=12)", "H2", h2_kp(7, 12, w, p)[0], 20)
check("Example H2 (c=12)", "Hl^2", hl_kp(7, 12, w, p, 2)[0], 20)
check("Example H2 (c=12)", "Hl^3", hl_kp(7, 12, w, p, 3)[0], 20)

p = [2, 3, 3, 3, 3]
w = [1, 3, 3, 3, 3]
check("Example H^l", "GR", greedy_kp(5, 12, w, p)[0], 11)
check("Example H^l", "GRe", extended_greedy_kp(5, 12, w, p)[0], 11)
check("Example H^l", "H2", h2_kp(5, 12, w, p)[0], 11)
check("Example H^l", "Hl^2", hl_kp(5, 12, w, p, 2)[0], 11)
check("Example H^l", "Hl^3", hl_kp(5, 12, w, p, 3)[0], 12)

p = [19, 19, 14, 14]
w = [2, 4, 3, 3]
check("Example FPTAS", "GR", greedy_kp(4, 8, w, p)[0], 38)
check("Example FPTAS", "GRe", extended_greedy_kp(4, 8, w, p)[0], 38)
check("Example FPTAS", "H2", h2_kp(4, 8, w, p)[0], 47)
check("Example FPTAS", "Hl^2", hl_kp(4, 8, w, p, 2)[0], 47)
check("Example FPTAS", "Hl^3", hl_kp(4, 8, w, p, 3)[0], 47)
check("Example FPTAS", "FP(eps=1/2)", basic_fptas(4, 8, w, p, 1, 2)[0], 38)

p = [12, 6, 6, 8, 6]
w = [3, 2, 3, 9, 7]
check("EX_GR_1", "GR", greedy_kp(5, 15, w, p)[0], 30)
check("EX_GR_1", "GRe", extended_greedy_kp(5, 15, w, p)[0], 30)
check("EX_GR_1", "H2", h2_kp(5, 15, w, p)[0], 30)
check("EX_GR_1", "Hl^2", hl_kp(5, 15, w, p, 2)[0], 30)
check("EX_GR_1", "Hl^3", hl_kp(5, 15, w, p, 3)[0], 30)

p = [10, 6, 3, 7, 11, 2]
w = [1, 1, 1, 4, 8, 3]
check("EX_GR_2", "GR", greedy_kp(6, 13, w, p)[0], 28)
check("EX_GR_2", "GRe", extended_greedy_kp(6, 13, w, p)[0], 28)
check("EX_GR_2", "H2", h2_kp(6, 13, w, p)[0], 30)
check("EX_GR_2", "Hl^2", hl_kp(6, 13, w, p, 2)[0], 30)
check("EX_GR_2", "Hl^3", hl_kp(6, 13, w, p, 3)[0], 30)

p = [8, 7, 18, 14, 10]
w = [3, 3, 9, 8, 7]
check("EX_GR_3", "GR", greedy_kp(5, 12, w, p)[0], 15)
check("EX_GR_3", "GRe", extended_greedy_kp(5, 12, w, p)[0], 18)
check("EX_GR_3", "H2", h2_kp(5, 12, w, p)[0], 26)
check("EX_GR_3", "Hl^2", hl_kp(5, 12, w, p, 2)[0], 26)
check("EX_GR_3", "Hl^3", hl_kp(5, 12, w, p, 3)[0], 26)

p = [11, 5, 3, 4, 3]
w = [8, 4, 3, 9, 8]
check("EX_GRe_1", "GR", greedy_kp(5, 22, w, p)[0], 19)
check("EX_GRe_1", "GRe", extended_greedy_kp(5, 22, w, p)[0], 19)
check("EX_GRe_1", "H2", h2_kp(5, 22, w, p)[0], 20)
check("EX_GRe_1", "Hl^2", hl_kp(5, 22, w, p, 2)[0], 20)
check("EX_GRe_1", "Hl^3", hl_kp(5, 22, w, p, 3)[0], 20)

p = [16, 19, 18, 14, 8]
w = [5, 8, 8, 7, 6]
check("EX_GRe_2", "GR", greedy_kp(5, 8, w, p)[0], 16)
check("EX_GRe_2", "GRe", extended_greedy_kp(5, 8, w, p)[0], 19)
check("EX_GRe_2", "H2", h2_kp(5, 8, w, p)[0], 19)
check("EX_GRe_2", "Hl^2", hl_kp(5, 8, w, p, 2)[0], 19)
check("EX_GRe_2", "Hl^3", hl_kp(5, 8, w, p, 3)[0], 19)

p = [2, 8, 8]
w = [1, 8, 8]
check("EX_GRe_3", "GR", greedy_kp(3, 16, w, p)[0], 10)
check("EX_GRe_3", "GRe", extended_greedy_kp(3, 16, w, p)[0], 10)
check("EX_GRe_3", "H2", h2_kp(3, 16, w, p)[0], 16)
check("EX_GRe_3", "Hl^2", hl_kp(3, 16, w, p, 2)[0], 16)
check("EX_GRe_3", "Hl^3", hl_kp(3, 16, w, p, 3)[0], 16)

p = [8, 9, 2, 3, 2]
w = [5, 8, 3, 7, 9]
check("EX_H2_1", "GR", greedy_kp(5, 11, w, p)[0], 10)
check("EX_H2_1", "GRe", extended_greedy_kp(5, 11, w, p)[0], 10)
check("EX_H2_1", "H2", h2_kp(5, 11, w, p)[0], 11)
check("EX_H2_1", "Hl^2", hl_kp(5, 11, w, p, 2)[0], 11)
check("EX_H2_1", "Hl^3", hl_kp(5, 11, w, p, 3)[0], 11)

p = [14, 9, 5, 10, 15]
w = [1, 1, 2, 5, 8]
check("EX_H2_2", "GR", greedy_kp(5, 15, w, p)[0], 38)
check("EX_H2_2", "GRe", extended_greedy_kp(5, 15, w, p)[0], 38)
check("EX_H2_2", "H2", h2_kp(5, 15, w, p)[0], 43)
check("EX_H2_2", "Hl^2", hl_kp(5, 15, w, p, 2)[0], 43)
check("EX_H2_2", "Hl^3", hl_kp(5, 15, w, p, 3)[0], 48)

p = [13, 5, 8, 5, 7]
w = [1, 1, 3, 2, 5]
check("EX_H2_3", "GR", greedy_kp(5, 10, w, p)[0], 31)
check("EX_H2_3", "GRe", extended_greedy_kp(5, 10, w, p)[0], 31)
check("EX_H2_3", "H2", h2_kp(5, 10, w, p)[0], 31)
check("EX_H2_3", "Hl^2", hl_kp(5, 10, w, p, 2)[0], 31)
check("EX_H2_3", "Hl^3", hl_kp(5, 10, w, p, 3)[0], 33)

p = [14, 13, 13, 10, 11]
w = [1, 2, 3, 3, 9]
check("EX_Hl_1", "GR", greedy_kp(5, 16, w, p)[0], 50)
check("EX_Hl_1", "GRe", extended_greedy_kp(5, 16, w, p)[0], 50)
check("EX_Hl_1", "H2", h2_kp(5, 16, w, p)[0], 50)
check("EX_Hl_1", "Hl^2", hl_kp(5, 16, w, p, 2)[0], 50)
check("EX_Hl_1", "Hl^3", hl_kp(5, 16, w, p, 3)[0], 51)

p = [2, 4, 4, 4, 4, 4]
w = [1, 4, 4, 4, 4, 4]
check("EX_Hl_2", "GR", greedy_kp(6, 20, w, p)[0], 18)
check("EX_Hl_2", "GRe", extended_greedy_kp(6, 20, w, p)[0], 18)
check("EX_Hl_2", "H2", h2_kp(6, 20, w, p)[0], 18)
check("EX_Hl_2", "Hl^2", hl_kp(6, 20, w, p, 2)[0], 18)
check("EX_Hl_2", "Hl^3", hl_kp(6, 20, w, p, 3)[0], 18)

p = [15, 15, 14, 8, 12, 13]
w = [1, 2, 3, 2, 6, 9]
check("EX_Hl_3", "GR", greedy_kp(6, 18, w, p)[0], 64)
check("EX_Hl_3", "GRe", extended_greedy_kp(6, 18, w, p)[0], 64)
check("EX_Hl_3", "H2", h2_kp(6, 18, w, p)[0], 64)
check("EX_Hl_3", "Hl^2", hl_kp(6, 18, w, p, 2)[0], 64)
check("EX_Hl_3", "Hl^3", hl_kp(6, 18, w, p, 3)[0], 64)

p = [24, 11, 23, 5]
w = [2, 3, 9, 3]
check("EX_FP_1", "GR", greedy_kp(4, 13, w, p)[0], 40)
check("EX_FP_1", "GRe", extended_greedy_kp(4, 13, w, p)[0], 40)
check("EX_FP_1", "H2", h2_kp(4, 13, w, p)[0], 47)
check("EX_FP_1", "Hl^2", hl_kp(4, 13, w, p, 2)[0], 47)
check("EX_FP_1", "Hl^3", hl_kp(4, 13, w, p, 3)[0], 47)
check("EX_FP_1", "FP(eps=1/2)", basic_fptas(4, 13, w, p, 1, 2)[0], 47)
check("EX_FP_1", "FP(eps=1/4)", basic_fptas(4, 13, w, p, 1, 4)[0], 47)

p = [14, 12, 14, 18]
w = [3, 4, 5, 7]
check("EX_FP_2", "GR", greedy_kp(4, 16, w, p)[0], 40)
check("EX_FP_2", "GRe", extended_greedy_kp(4, 16, w, p)[0], 40)
check("EX_FP_2", "H2", h2_kp(4, 16, w, p)[0], 46)
check("EX_FP_2", "Hl^2", hl_kp(4, 16, w, p, 2)[0], 46)
check("EX_FP_2", "Hl^3", hl_kp(4, 16, w, p, 3)[0], 46)
check("EX_FP_2", "FP(eps=1/2)", basic_fptas(4, 16, w, p, 1, 2)[0], 44)
check("EX_FP_2", "FP(eps=1/4)", basic_fptas(4, 16, w, p, 1, 4)[0], 46)

p = [14, 12, 14, 18]
w = [3, 4, 5, 7]
check("EX_FP_3", "GR", greedy_kp(4, 16, w, p)[0], 40)
check("EX_FP_3", "GRe", extended_greedy_kp(4, 16, w, p)[0], 40)
check("EX_FP_3", "H2", h2_kp(4, 16, w, p)[0], 46)
check("EX_FP_3", "Hl^2", hl_kp(4, 16, w, p, 2)[0], 46)
check("EX_FP_3", "Hl^3", hl_kp(4, 16, w, p, 3)[0], 46)
check("EX_FP_3", "FP(eps=1/2)", basic_fptas(4, 16, w, p, 1, 2)[0], 44)
check("EX_FP_3", "FP(eps=1/4)", basic_fptas(4, 16, w, p, 1, 4)[0], 46)

print("\n%d checks, %d errors" % (checks, errors))

Example 1 (c=9)        GR               14 OK
Example 1 (c=9)        GRe              14 OK
Example 1 (c=9)        H2               15 OK
Example 1 (c=9)        Hl^2             15 OK
Example 1 (c=9)        Hl^3             15 OK
Example 2              GR               18 OK
Example 2              GRe              19 OK
Example 2              H2               19 OK
Example 2              Hl^2             19 OK
Example 2              Hl^3             19 OK
Example 3              GR               18 OK
Example 3              GRe              18 OK
Example 3              H2               23 OK
Example 3              Hl^2             23 OK
Example 3              Hl^3             23 OK
Example 4              GR               19 OK
Example 4              GRe              20 OK
Example 4              H2               20 OK
Example 4              Hl^2             20 OK
Example 4              Hl^3             20 OK
Example H2 (c=12)      GR               19 OK
Example H2 (c=12)      GRe        

## Try it yourself

Change the data in the cell below and run it again.

In [9]:
# Try it yourself: change the data below and run the cell again
# (the items must be sorted by non-increasing efficiency p[i]/w[i], as the functions assume)
c = 9                         # knapsack capacity
p = [6, 5, 8, 9, 6, 7, 3]     # profits
w = [2, 3, 6, 7, 5, 9, 4]     # weights
l = 3                         # number of guessed items of H^l (2 <= l <= n)
eps_num, eps_den = 1, 2       # accuracy eps = eps_num / eps_den of the Basic-FPTAS

n = len(p)
results = [("GR", greedy_kp(n, c, w, p)),
           ("GR^e", extended_greedy_kp(n, c, w, p)),
           ("H^2", h2_kp(n, c, w, p)),
           ("H^%d" % l, hl_kp(n, c, w, p, l)),
           ("FPTAS eps=%d/%d" % (eps_num, eps_den), basic_fptas(n, c, w, p, eps_num, eps_den)),
           ("optimum (DP)", dp_profits(n, c, w, p, sum(p)))]
print("Instance: c = %d, n = %d items" % (c, n))
for name, (z, x) in results:
    line = "%-15s z = %3d   x =" % (name, z)
    for i in range(n):
        line += " %d" % x[i]
    print(line)

Instance: c = 9, n = 7 items
GR              z =  14   x = 1 1 0 0 0 0 1
GR^e            z =  14   x = 1 1 0 0 0 0 1
H^2             z =  15   x = 1 0 0 1 0 0 0
H^3             z =  15   x = 1 0 0 1 0 0 0
FPTAS eps=1/2   z =  15   x = 1 0 0 1 0 0 0
optimum (DP)    z =  15   x = 1 0 0 1 0 0 0


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).